In [1]:
import torch
import opfunu
import numpy as np
from algorithms.pso import PSO
from algorithms.ga import GA
from algorithms.cmaes import CMAES
from algorithms.de import DE

from fstpso import FuzzyPSO
from algorithms.minimize import minimize

In [2]:
def make_opfunu_autograd(opfunu_instance, eps=1e-6, parallel=True):
    """
    Turn ANY opfunu benchmark instance into a differentiable torch objective.

    Parameters
    ----------
    opfunu_instance : e.g. opfunu.name_based.Ackley01(dim)
    eps             : finite-difference stepsize
    parallel        : True -> vectorised finite diff (faster, more mem)

    Returns
    -------
    obj(pop : torch.Tensor[N,D]) -> torch.Tensor[N]
         differentiable w.r.t. pop  (gradient is FD estimate)
    """
    fn = opfunu_instance        # keep reference in closure
    D  = fn.ndim

    class _OpfunuFD(torch.autograd.Function):

        @staticmethod
        def forward(ctx, x):
            # x : [N, D] tensor (CPU or CUDA)
            x_cpu = x.detach().cpu().numpy()
            vals  = np.array([fn.evaluate(p) for p in x_cpu],
                             dtype=np.float32)          # shape [N]
            out   = torch.tensor(vals, device=x.device, dtype=x.dtype)
            ctx.save_for_backward(x)                    # for backward
            return out

        @staticmethod
        def backward(ctx, grad_out):
            (x,)   = ctx.saved_tensors                  # [N,D] on same device
            N, D_  = x.shape
            device = x.device
            x_cpu  = x.detach().cpu().numpy()

            if parallel:  # vectorised FD (2·D evaluations at once)
                eye   = np.eye(D_, dtype=np.float32) * eps
                Xplus = (x_cpu[:, None, :] + eye).reshape(-1, D_)
                Xminus= (x_cpu[:, None, :] - eye).reshape(-1, D_)
                batch = np.vstack([Xplus, Xminus])      # shape [2·N·D, D]

                vals  = np.array([fn.evaluate(p) for p in batch],
                                 dtype=np.float32).reshape(2, N, D_)
                grad  = (vals[0] - vals[1]) / (2*eps)   # [N,D]
                grad  = torch.tensor(grad, device=device, dtype=x.dtype)

            else:       # loop over dims (less RAM, slower)
                grad = torch.zeros_like(x)
                for d in range(D_):
                    e = np.zeros_like(x_cpu)
                    e[:, d] += eps
                    f_plus = np.array([fn.evaluate(p) for p in x_cpu+e],
                                      dtype=np.float32)
                    f_minus= np.array([fn.evaluate(p) for p in x_cpu-e],
                                      dtype=np.float32)
                    grad[:, d] = torch.tensor((f_plus - f_minus)/(2*eps),
                                              device=device, dtype=x.dtype)
            # chain rule
            return grad_out.unsqueeze(-1) * grad

    def objective(pop):
        return _OpfunuFD.apply(pop)

    return objective

In [3]:
def make_opfunu_fst_pso(opfunu_instance):
    fn = opfunu_instance
    def obj(p):
        return fn.evaluate(np.array(p))
    return obj

In [4]:
device = "cpu" 

if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda" 

device

'mps'

In [5]:
dim = 10
pop_size  = 100
max_evals = 1000*dim

In [6]:
# 1) pick any opfunu function and dimension
ackley10 = opfunu.name_based.Ackley01(dim)

# 2) wrap it for autograd and fst-pso
ackley_autograd = make_opfunu_autograd(ackley10, eps=1e-6)
ackley_fst_pso  = make_opfunu_fst_pso(ackley10)

lower_bound = ackley10.lb.astype(np.float32)
upper_bound = ackley10.ub.astype(np.float32)

In [7]:
pso = PSO(ackley_autograd, dim=dim, pop_size=pop_size, lower_bound=lower_bound, upper_bound=upper_bound, device=device)
minimize(pso, max_evals=max_evals, verbose=True)

pso.history["best_f"][-1]

Generation    1 | Loss = 16.7110, best_f = 16.7110
Generation    2 | Loss = 12.6267, best_f = 12.6267
Generation    3 | Loss = 12.8621, best_f = 12.6267
Generation    4 | Loss = 11.9014, best_f = 11.9014
Generation    5 | Loss = 11.5950, best_f = 11.5950
Generation    6 | Loss = 12.3427, best_f = 11.5950
Generation    7 | Loss = 11.4986, best_f = 11.4986
Generation    8 | Loss = 8.7382, best_f = 8.7382
Generation    9 | Loss = 8.9150, best_f = 8.7382
Generation   10 | Loss = 8.7068, best_f = 8.7068
Generation   11 | Loss = 7.5968, best_f = 7.5968
Generation   12 | Loss = 7.7933, best_f = 7.5968
Generation   13 | Loss = 6.7182, best_f = 6.7182
Generation   14 | Loss = 6.7249, best_f = 6.7182
Generation   15 | Loss = 6.5436, best_f = 6.5436
Generation   16 | Loss = 5.9692, best_f = 5.9692
Generation   17 | Loss = 5.7061, best_f = 5.7061
Generation   18 | Loss = 4.9902, best_f = 4.9902
Generation   19 | Loss = 5.4728, best_f = 4.9902
Generation   20 | Loss = 4.7422, best_f = 4.7422
Genera

0.0016566148260608315

In [8]:
FP = FuzzyPSO()
FP.set_search_space(list(zip(lower_bound, upper_bound)))
FP.set_swarm_size(pop_size)
FP.set_fitness(ackley_fst_pso)
FP.max_evaluations = max_evals
result = FP.solve_with_fstpso()

Fuzzy Self-Tuning PSO - v1.8.1
 * Max distance: 221.359436
 * Search space boundaries set to: [(np.float32(-35.0), np.float32(35.0)), (np.float32(-35.0), np.float32(35.0)), (np.float32(-35.0), np.float32(35.0)), (np.float32(-35.0), np.float32(35.0)), (np.float32(-35.0), np.float32(35.0)), (np.float32(-35.0), np.float32(35.0)), (np.float32(-35.0), np.float32(35.0)), (np.float32(-35.0), np.float32(35.0)), (np.float32(-35.0), np.float32(35.0)), (np.float32(-35.0), np.float32(35.0))]
 * Max velocities set to: [70.0, 70.0, 70.0, 70.0, 70.0, 70.0, 70.0, 70.0, 70.0, 70.0]
 * Number of particles automatically set to 16
 * Swarm size now set to 100 particles
 * 100 particles created.
 * FST-PSO will now assess the local and global best particles.
 * Estimated worst fitness: 21.809
 * New best particle in the swarm is #0 with fitness 21.784898 (it: 0).
 * New best particle in the swarm is #1 with fitness 21.522262 (it: 0).
 * New best particle in the swarm is #3 with fitness 21.311015 (it: 0).
 

In [9]:
# 1) pick any opfunu function and dimension
function = opfunu.name_based.Levy03(dim)

# 2) wrap it for autograd and fst-pso
function_autograd = make_opfunu_autograd(function, eps=1e-6)
function_fst_pso  = make_opfunu_fst_pso(function)

lower_bound = function.lb.astype(np.float32)
upper_bound = function.ub.astype(np.float32)

In [10]:
pso = PSO(function_autograd, dim=dim, pop_size=pop_size, lower_bound=lower_bound, upper_bound=upper_bound, device=device)
minimize(pso, max_evals=max_evals, verbose=True)

pso.history["best_f"][-1]

Generation    1 | Loss = 7.6837, best_f = 7.6837
Generation    2 | Loss = 6.8100, best_f = 6.8100
Generation    3 | Loss = 4.4871, best_f = 4.4871
Generation    4 | Loss = 4.2159, best_f = 4.2159
Generation    5 | Loss = 3.9577, best_f = 3.9577
Generation    6 | Loss = 3.1837, best_f = 3.1837
Generation    7 | Loss = 2.3571, best_f = 2.3571
Generation    8 | Loss = 2.1071, best_f = 2.1071
Generation    9 | Loss = 0.9795, best_f = 0.9795
Generation   10 | Loss = 0.7469, best_f = 0.7469
Generation   11 | Loss = 0.6924, best_f = 0.6924
Generation   12 | Loss = 1.1018, best_f = 0.6924
Generation   13 | Loss = 0.6149, best_f = 0.6149
Generation   14 | Loss = 0.7858, best_f = 0.6149
Generation   15 | Loss = 0.7300, best_f = 0.6149
Generation   16 | Loss = 0.6788, best_f = 0.6149
Generation   17 | Loss = 0.7049, best_f = 0.6149
Generation   18 | Loss = 0.3830, best_f = 0.3830
Generation   19 | Loss = 0.2750, best_f = 0.2750
Generation   20 | Loss = 0.2069, best_f = 0.2069
Generation   21 | Lo

2.3086156986096285e-08

In [11]:
FP = FuzzyPSO()
FP.set_search_space(list(zip(lower_bound, upper_bound)))
FP.set_swarm_size(pop_size)
FP.set_fitness(function_fst_pso)
FP.max_evaluations = max_evals
result = FP.solve_with_fstpso()

Fuzzy Self-Tuning PSO - v1.8.1
 * Max distance: 63.245553
 * Search space boundaries set to: [(np.float32(-10.0), np.float32(10.0)), (np.float32(-10.0), np.float32(10.0)), (np.float32(-10.0), np.float32(10.0)), (np.float32(-10.0), np.float32(10.0)), (np.float32(-10.0), np.float32(10.0)), (np.float32(-10.0), np.float32(10.0)), (np.float32(-10.0), np.float32(10.0)), (np.float32(-10.0), np.float32(10.0)), (np.float32(-10.0), np.float32(10.0)), (np.float32(-10.0), np.float32(10.0))]
 * Max velocities set to: [20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0]
 * Number of particles automatically set to 16
 * Swarm size now set to 100 particles
 * 100 particles created.
 * FST-PSO will now assess the local and global best particles.
 * Estimated worst fitness: 335.888
 * New best particle in the swarm is #0 with fitness 154.812256 (it: 0).
 * New best particle in the swarm is #1 with fitness 46.682739 (it: 0).
 * New best particle in the swarm is #64 with fitness 38.672630 (it: 0).

/Users/tangherloni/anaconda3/envs/AI/lib/python3.11/site-packages/fstpso/fstpso.py:382: RuntimeWarning: overflow encountered in cast
  if self.Solutions[i].CalculatedFitness < self.Solutions[i].CalculatedBestFitness:
/Users/tangherloni/anaconda3/envs/AI/lib/python3.11/site-packages/fstpso/fstpso.py:388: RuntimeWarning: overflow encountered in cast
  if self.Solutions[i].CalculatedFitness < self.G.CalculatedFitness:


 * New best particle in the swarm is #34 with fitness 0.001242 (it: 22).
 * New best particle in the swarm is #38 with fitness 0.001079 (it: 24).
 * 25th iteration out of 100 completed. [#######                       ]
 * New best particle in the swarm is #38 with fitness 0.000988 (it: 25).
 * New best particle in the swarm is #92 with fitness 0.000964 (it: 25).
 * New best particle in the swarm is #45 with fitness 0.000942 (it: 26).
 * New best particle in the swarm is #72 with fitness 0.000881 (it: 26).
 * New best particle in the swarm is #16 with fitness 0.000849 (it: 28).
 * New best particle in the swarm is #34 with fitness 0.000554 (it: 28).
 * New best particle in the swarm is #61 with fitness 0.000490 (it: 28).
 * New best particle in the swarm is #72 with fitness 0.000389 (it: 29).
 * New best particle in the swarm is #16 with fitness 0.000246 (it: 30).
 * New best particle in the swarm is #49 with fitness 0.000185 (it: 31).
 * New best particle in the swarm is #38 with fitne

In [12]:
cmaes = CMAES(function_autograd, dim=dim, pop_size=pop_size, lower_bound=lower_bound, upper_bound=upper_bound, device = "cpu")
minimize(cmaes, max_evals=max_evals, verbose=True)

Generation    1 | Loss = 1.8554, best_f = 1.8554
Generation    2 | Loss = 0.6397, best_f = 0.6397
Generation    3 | Loss = 0.4237, best_f = 0.4237
Generation    4 | Loss = 0.2277, best_f = 0.2277
Generation    5 | Loss = 0.2620, best_f = 0.2277
Generation    6 | Loss = 0.4021, best_f = 0.2277
Generation    7 | Loss = 0.3269, best_f = 0.2277
Generation    8 | Loss = 0.2837, best_f = 0.2277
Generation    9 | Loss = 0.1041, best_f = 0.1041
Generation   10 | Loss = 0.1400, best_f = 0.1041
Generation   11 | Loss = 0.0393, best_f = 0.0393
Generation   12 | Loss = 0.0638, best_f = 0.0393
Generation   13 | Loss = 0.0226, best_f = 0.0226
Generation   14 | Loss = 0.0227, best_f = 0.0226
Generation   15 | Loss = 0.0081, best_f = 0.0081
Generation   16 | Loss = 0.0089, best_f = 0.0081
Generation   17 | Loss = 0.0134, best_f = 0.0081
Generation   18 | Loss = 0.0033, best_f = 0.0033
Generation   19 | Loss = 0.0056, best_f = 0.0033
Generation   20 | Loss = 0.0035, best_f = 0.0033
Generation   21 | Lo

In [13]:
def ackley(x, a=20, b=0.2, c=2 * torch.pi):
    """
    Compute the Ackley function.

    Parameters:
        x (torch.Tensor): Input tensor of shape [batch_size, dim].
        a (float): Parameter a (default=20).
        b (float): Parameter b (default=0.2).
        c (float): Parameter c (default=2*pi).

    Returns:
        torch.Tensor: A tensor of shape [batch_size] with the Ackley function values.
    """
    dim = x.size(-1)
    sum_sq_term = torch.sum(x ** 2, dim=-1) / dim
    cos_term = torch.sum(torch.cos(c * x), dim=-1) / dim

    term1 = -a * torch.exp(-b * torch.sqrt(sum_sq_term))
    term2 = -torch.exp(cos_term)

    # torch.exp(torch.tensor(1.0)) is used to compute e^1.
    return term1 + term2 + a + torch.exp(torch.tensor(1.0, device=x.device))

def Plateau(x):
    if x.dim() == 1:
        return 30.0 + torch.sum(torch.floor(x))
    elif x.dim() == 2:
        return 30.0 + torch.sum(torch.floor(x), dim=1)
    

def Vincent(x):
    if x.dim() == 1:
        x = x.unsqueeze(0)
    
    epsilon = 1e-10
    x = torch.clamp(x, min=epsilon)

    d = x.size(1)     
    return (1.0 / d) * torch.sum(torch.sin(10 * torch.log(x)), dim=-1)

In [14]:
# dim = 10
# lower_bound = [-10.0]*dim
# upper_bound = [5.0]*dim
# pop_size  = 100

# max_evals = 1000*dim

In [15]:
# ga = GA(ackley, dim=dim, pop_size=pop_size, lower_bound=lower_bound, upper_bound=upper_bound)
# minimize(ga, max_evals=max_evals, verbose=True)

# ga.history["best_f"][-1]

In [16]:
# de = DE(ackley, dim=dim, pop_size=pop_size, lower_bound=lower_bound, upper_bound=upper_bound)
# minimize(de, max_evals=max_evals*dim, verbose=True)

# de.history["best_f"][-1]

In [17]:
pso = PSO(ackley, dim=dim, pop_size=pop_size, lower_bound=lower_bound, upper_bound=upper_bound)
minimize(pso, max_evals=max_evals, verbose=True)

pso.history["best_f"][-1]

Generation    1 | Loss = 8.0578, best_f = 8.0578
Generation    2 | Loss = 5.5949, best_f = 5.5949
Generation    3 | Loss = 5.4562, best_f = 5.4562
Generation    4 | Loss = 4.9934, best_f = 4.9934
Generation    5 | Loss = 5.1886, best_f = 4.9934
Generation    6 | Loss = 4.5550, best_f = 4.5550
Generation    7 | Loss = 4.4342, best_f = 4.4342
Generation    8 | Loss = 4.5005, best_f = 4.4342
Generation    9 | Loss = 4.6912, best_f = 4.4342
Generation   10 | Loss = 4.7014, best_f = 4.4342
Generation   11 | Loss = 3.6616, best_f = 3.6616
Generation   12 | Loss = 3.5940, best_f = 3.5940
Generation   13 | Loss = 4.0852, best_f = 3.5940
Generation   14 | Loss = 3.8291, best_f = 3.5940
Generation   15 | Loss = 3.2184, best_f = 3.2184
Generation   16 | Loss = 3.4014, best_f = 3.2184
Generation   17 | Loss = 3.2461, best_f = 3.2184
Generation   18 | Loss = 2.9123, best_f = 2.9123
Generation   19 | Loss = 1.8979, best_f = 1.8979
Generation   20 | Loss = 2.2652, best_f = 1.8979
Generation   21 | Lo

0.00035381317138671875

In [18]:
pso.device

'mps'

In [19]:
cmaes = CMAES(ackley, dim=dim, pop_size=pop_size, lower_bound=lower_bound, upper_bound=upper_bound, device = "cpu" )
minimize(cmaes, max_evals=max_evals, verbose=True)

cmaes.history["best_f"][-1]

Generation    1 | Loss = 2.9346, best_f = 2.9346
Generation    2 | Loss = 2.2248, best_f = 2.2248
Generation    3 | Loss = 2.2011, best_f = 2.2011
Generation    4 | Loss = 2.3606, best_f = 2.2011
Generation    5 | Loss = 1.9782, best_f = 1.9782
Generation    6 | Loss = 2.1128, best_f = 1.9782
Generation    7 | Loss = 2.1428, best_f = 1.9782
Generation    8 | Loss = 2.1022, best_f = 1.9782
Generation    9 | Loss = 1.6590, best_f = 1.6590
Generation   10 | Loss = 1.4816, best_f = 1.4816
Generation   11 | Loss = 1.0791, best_f = 1.0791
Generation   12 | Loss = 1.0822, best_f = 1.0791
Generation   13 | Loss = 0.4975, best_f = 0.4975
Generation   14 | Loss = 0.6759, best_f = 0.4975
Generation   15 | Loss = 0.3278, best_f = 0.3278
Generation   16 | Loss = 0.3239, best_f = 0.3239
Generation   17 | Loss = 0.3739, best_f = 0.3239
Generation   18 | Loss = 0.1705, best_f = 0.1705
Generation   19 | Loss = 0.2318, best_f = 0.1705
Generation   20 | Loss = 0.1762, best_f = 0.1705
Generation   21 | Lo

7.343292236328125e-05

In [20]:
cmaes.device

'cpu'